### this is a development environment for decoding code. Once it works here, throw it into the main function in decoding_dcc.py.

set up imports and testing variables (all lowercase - note these are the variables from run_decoding_dcc.py but defined here rather than imported)

In [7]:
import sys
import os
import numpy as np
import pandas as pd
import json
import pickle
from functools import partial
from datetime import datetime
from types import SimpleNamespace

import mne
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from joblib import Parallel, delayed
from scipy.stats import ttest_ind, ttest_rel
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC

from ieeg.navigate import channel_outlier_marker, trial_ieeg, crop_empty_data, outliers_to_nan
from ieeg.io import raw_from_layout, get_data
from ieeg.calc.stats import time_perm_cluster
from ieeg.calc.fast import mean_diff
from ieeg.decoding.models import PcaLdaClassification
from ieeg.calc.oversample import MinimumNaNSplit

try:
    current_file_path = os.path.abspath(__file__)
    current_script_dir = os.path.dirname(current_file_path)
except NameError:
    current_script_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_script_dir, '..', '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.analysis.config import experiment_conditions
from src.analysis.config.condition_registry import get_comparisons

from src.analysis.utils.general_utils import (
    make_or_load_subjects_electrodes_to_ROIs_dict,
    load_subjects_electrodes_to_ROIs_dict,
    identify_bad_channels_by_trial_nan_rate,
    impute_trial_nans_by_channel_mean,
    create_subjects_mne_objects_dict,
    filter_electrode_lists_against_subjects_mne_objects,
    find_difference_between_two_electrode_lists,
    print_summary_of_dropped_electrodes,
    get_conditions_save_name,
    get_default_LAB_root,
    get_sig_chans_per_subject,
    make_sig_electrodes_per_subject_and_roi_dict,
)

from src.analysis.utils.labeled_array_utils import (
    put_data_in_labeled_array_per_roi_subject,
    remove_nans_from_labeled_array,
    remove_nans_from_all_roi_labeled_arrays,
    concatenate_conditions_by_string,
    get_data_in_time_range,
    make_bootstrapped_roi_labeled_arrays_with_nan_trials_removed_for_each_channel,
)

from src.analysis.decoding.decoding import (
    get_and_plot_confusion_matrix_for_rois_jim,
    Decoder,
    windower,
    get_confusion_matrices_for_rois_time_window_decoding_jim,
    compute_accuracies,
    plot_true_vs_shuffle_accuracies,
    plot_accuracies_with_multiple_sig_clusters,
    plot_accuracies_nature_style,
    make_pooled_shuffle_distribution,
    find_significant_clusters_of_series_vs_distribution_based_on_percentile,
    compute_pooled_bootstrap_statistics,
    do_time_perm_cluster_comparing_two_true_bootstrap_accuracy_distributions,
    do_mne_paired_cluster_test,
    get_pooled_accuracy_distributions_for_comparison,
    get_time_averaged_confusion_matrix,
    cluster_perm_paired_ttest_by_duration,
    run_two_one_tailed_tests_with_time_perm_cluster,
    extract_pooled_cm_traces,
    plot_cm_traces_nature_style,
    run_context_comparison_analysis,
    plot_cross_block_overlay,
)

from src.analysis.decoding.process_bootstrap import process_bootstrap
from src.analysis.decoding.run_visualization_debug import run_visualization_debug
from src.analysis.decoding.run_debug_cm_traces import run_debug_cm_traces
from src.analysis.decoding.run_aggregate_and_plot_time_averaged_cms import run_aggregate_and_plot_time_averaged_cms
from src.analysis.decoding.run_context_comparisons import run_all_context_comparisons

# task
task = 'GlobalLocal'

# trial selection
# switched to False for err-corr decoding
acc_trials_only = True

# decoding parameters
# first, choose your classifier
model_choice = 'LDA'  # options: 'LDA', 'SVC'

if model_choice == 'LDA':
    clf_model = LinearDiscriminantAnalysis()
    clf_model_str = 'LDA'
elif model_choice == 'SVC':
    # you can configure your model here
    clf_model = SVC(C=1.0, kernel='linear', probability=False)
    clf_model_str = 'SVC_C1_linear'
else:
    raise ValueError(f"Unknown model_choice: {model_choice}")

# then, choose your decoding parameters
random_state = 42
explained_variance = 0.90
balance_method = 'subsample'
normalize = 'true'
obs_axs = 0
chans_axs = 1
time_axs = -1

# time-windowed decoding parameters
window_size = 64  # window size in samples (e.g., 64 samples = 250 ms at 256 Hz)
step_size = 16    # step size in samples (e.g., 16 samples = 62.5 ms at 256 Hz)
sampling_rate = 256  # sampling rate of the data in Hz
first_time_point = -1.5  # the time in seconds of the first sample in the epoch
tails = 1  # 1 for one-tailed (e.g., accuracy > chance), 2 for two-tailed
n_shuffle_perms = 2  # how many times to shuffle labels and train decoder to make chance decoding results - this iterates over splits, so end up with n_shuffle_perms * n_splits for number of folds

# whether to do stats across fold, repeat, or bootstrap
unit_of_analysis = 'repeat'

# whether to store individual folds (true) or sum them within repeats (false)
folds_as_samples = True if unit_of_analysis == 'fold' else False

# percentile stats parameters
percentile = 95
cluster_percentile = 95

# additional parameters for permutation cluster stats
stat_func_choice = 'ttest_ind'  # 'ttest_ind', 'ttest_rel' or 'mean_diff'

if stat_func_choice == 'mean_diff':
    stat_func = mean_diff
    stat_func_str = 'mean_diff'
elif stat_func_choice == 'ttest_ind':
    stat_func = partial(ttest_ind, equal_var=False, nan_policy='omit')
    stat_func_str = 'ttest_ind'
elif stat_func_choice == 'ttest_rel':
    stat_func = partial(ttest_rel, nan_policy='omit')
    stat_func_str = 'ttest_rel'

p_thresh_for_time_perm_cluster_stats = 0.025
p_cluster = 0.025
permutation_type = 'independent'
# cluster_tails = 2

# plotting
single_column = True
show_legend = False
run_visualization_debug = False  # collapsed onto the first two PCs, this plots each trial and the SVM or LDA hyperplane.

condition_name = 'stimulus_lwpc_conditions'
conditions = getattr(experiment_conditions, condition_name)
condition_label = condition_name  # pass this into args for output path naming

# epochs file selection - just use this one for testing
epochs_root_file = "Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit"

# which electrodes to use (all or sig) - TODO: add in an option to include a dictionary of electrodes here, like congruencySigElectrodes
electrodes = 'sig'

# # # # testing params (comment out)
subjects = ['D0103']
n_splits = 2
n_repeats = 1
n_perm = 2
n_cluster_perms = 2
bootstraps = 2
n_jobs = -1
rois_dict = {
     'dlpfc': ["G_front_middle", "G_front_sup", "S_front_inf", "S_front_middle", "S_front_sup"],
}


args = SimpleNamespace(
    # Bootstrap control
    bootstraps=bootstraps,
    random_state=random_state,
    subjects=subjects,

    # Axes
    obs_axs=obs_axs,
    chans_axs=chans_axs,
    time_axs=time_axs,

    # Decoder / CV
    clf_model=clf_model,
    n_splits=n_splits,
    n_repeats=n_repeats,
    explained_variance=explained_variance,
    balance_method=balance_method,
    folds_as_samples=folds_as_samples,

    # Time-window decoding
    window_size=window_size,
    step_size=step_size,
    sampling_rate=sampling_rate,
    n_shuffle_perms=n_shuffle_perms,

    # Pooled shuffle
    conditions=conditions,  # the list from experiment_conditions.stimulus_lwpc_conditions
)

set up paths

In [8]:
# Determine LAB_root based on the operating system and environment
LAB_root = get_default_LAB_root()

print('LAB_root: ', LAB_root)

config_dir = os.path.join(project_root, 'src', 'analysis', 'config')
subjects_electrodestoROIs_dict = load_subjects_electrodes_to_ROIs_dict(save_dir=config_dir, filename='subjects_electrodestoROIs_dict.json')

condition_names = list(conditions.keys()) # get the condition names as a list

save_dir = os.path.join(LAB_root, 'BIDS-1.1_GlobalLocal', 'BIDS', 'derivatives', 'decoding', 'figs', f"{epochs_root_file}")
os.makedirs(save_dir, exist_ok=True)
print(f"Save directory created or already exists at: {save_dir}")

LAB_root:  /cwork/jz421
Loaded data from /hpc/group/coganlab/jz421/GlobalLocal/src/analysis/config/subjects_electrodestoROIs_dict.json
Save directory created or already exists at: /cwork/jz421/BIDS-1.1_GlobalLocal/BIDS/derivatives/decoding/figs/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit


load data (here again, all variables normally have args in front of them but dropping that prefix for testing since those variables are defined in this notebook now rather than imported from the run script)

In [9]:
sig_chans_per_subject = get_sig_chans_per_subject(subjects, epochs_root_file, task=task, LAB_root=LAB_root)

rois = list(rois_dict.keys())
all_electrodes_per_subject_roi, sig_electrodes_per_subject_roi = make_sig_electrodes_per_subject_and_roi_dict(rois_dict, subjects_electrodestoROIs_dict, sig_chans_per_subject)
    
subjects_mne_objects = create_subjects_mne_objects_dict(subjects=subjects, epochs_root_file=epochs_root_file, conditions=conditions, task=task, just_HG_ev1_rescaled=True, acc_trials_only=acc_trials_only)

# determine which electrodes to use (all electrodes or just the task-significant ones)
if electrodes == 'all':
    raw_electrodes = all_electrodes_per_subject_roi 
    elec_string_to_add_to_filename = 'all_elecs'
elif electrodes == 'sig':
    raw_electrodes = sig_electrodes_per_subject_roi
    elec_string_to_add_to_filename = 'sig_elecs'

else:
    raise ValueError("electrodes input must be set to all or sig")

# ADD THIS BLOCK to create a string for the sampling method
folds_info_str = 'folds_as_samples' if folds_as_samples else 'repeats_as_samples'

# filter electrodes to only the ones that exist in the epochs objects. This mismatch can arise due to dropping channels when making the epochs objects, because the subjects_electrodestoROIs_dict is made based on all the electrodes, with no dropping.
electrodes = filter_electrode_lists_against_subjects_mne_objects(rois, raw_electrodes, subjects_mne_objects)

print_summary_of_dropped_electrodes(raw_electrodes, electrodes)

condition_comparisons = get_comparisons(conditions)

# get the confusion matrix using the downsampled version
# add elec and subject info to filename 6/11/25
other_string_to_add = (
    f"{elec_string_to_add_to_filename}_{str(len(subjects))}_subjects_{folds_info_str}_ev_{explained_variance}"
)

Loaded significant channels for subject D0103
For subject D0057, G_front_middle, G_front_sup, S_front_inf, S_front_middle, S_front_sup electrodes are: ['RAI12', 'RAI13', 'RAI14', 'RAI15', 'RAI16', 'RPI15', 'RPI14', 'RAMF10', 'RAMF11', 'RAMF12', 'RAMF13', 'RAMF14']
For subject D0059, G_front_middle, G_front_sup, S_front_inf, S_front_middle, S_front_sup electrodes are: ['LMMF9', 'LMMF11', 'LMMF10', 'LMMF12', 'LPSF16']
For subject D0063, G_front_middle, G_front_sup, S_front_inf, S_front_middle, S_front_sup electrodes are: ['LASF10', 'LASF11', 'LASF14', 'LASF15', 'LASF16', 'LMSF5', 'LMSF6', 'LMSF12', 'LPSF10', 'LPSF12', 'ROF16', 'RAI16', 'RAMF11', 'RAMF12', 'RAMF13', 'RAMF14', 'RMMF13', 'RMMF14', 'RAI10', 'RAI11', 'RASF15', 'RASF16', 'RMSF8', 'RMSF9', 'RMSF10', 'RMSF11', 'RMSF12', 'RMSF7', 'RAMF8', 'RAMF9', 'RAMF10', 'RMMF9', 'RMMF10']
For subject D0065, G_front_middle, G_front_sup, S_front_inf, S_front_middle, S_front_sup electrodes are: ['RASF13', 'RASF14', 'RASF15', 'RASF16', 'RMSF11', 

/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:474: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


    Stimulus_c25: 159 valid trials out of 159
  Loading condition: Stimulus_c75
Adding metadata with 29 columns
41 matching events found
No baseline correction applied


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:474: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


    Stimulus_c75: 41 valid trials out of 41
  Loading condition: Stimulus_i25
Adding metadata with 29 columns
46 matching events found
No baseline correction applied


...

  Loading condition: Stimulus_i25
Adding metadata with 29 columns
46 matching events found
No baseline correction applied
    Stimulus_i25: 46 valid trials out of 46


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:474: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


  Loading condition: Stimulus_i75
Adding metadata with 29 columns
113 matching events found
No baseline correction applied


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:474: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


    Stimulus_i75: 113 valid trials out of 113

--- Summary of Dropped Electrodes ---
Total electrodes dropped across all subjects/ROIs: 0
-------------------------------------



try loading data using metadata instead - throw this into general_utils.py once it works

decoding (hmm refactor this to use the metadata labels instead 4/24)

In [ ]:
time_window_decoding_results = {}

# use joblib to run the bootstrap processing in parallel
bootstrap_results_list = Parallel(n_jobs=n_jobs, verbose=10, backend='loky')(
    delayed(process_bootstrap)(
        bootstrap_idx,
        subjects_mne_objects,
        args,
        rois,
        condition_names,
        electrodes,
        condition_comparisons,
        save_dir
    ) for bootstrap_idx in range(args.bootstraps)
)

In [ ]:

def main(args):
    
    # reconstruct the main results dictionary from the list returned by the parallel jobs
    time_window_decoding_results = {i: result['time_window_results'] for i, result in enumerate(bootstrap_results_list) if result is not None}
    time_averaged_cms_list = [result['time_averaged_cms'] for result in bootstrap_results_list if result]

    ## Extract the 'cats_by_roi' dictionary from the first valid bootstrap result.
    ## This is necessary to get the correct labels for plotting the confusion matrices.
    cats_by_roi = {}
    first_valid_result = next((res for res in bootstrap_results_list if res), None)
    if first_valid_result:
        cats_by_roi = first_valid_result.get('cats_by_roi', {})

    # --- Step 1: Aggregate and Plot Time-Averaged CMs ---
    run_aggregate_and_plot_time_averaged_cms(
        time_averaged_cms_list, condition_comparisons, rois, cats_by_roi, args, save_dir
    )
    
    if not time_window_decoding_results:
        print("\n✗ Analysis failed: No bootstrap samples were successfully processed.")
        return
    
    print(f"\n{'-'*20} PARALLEL BOOTSTRAPPING COMPLETE {'='*20}\n")
    
    # after all bootstraps complete, run pooled statistics
    all_bootstrap_stats = compute_pooled_bootstrap_statistics(
        time_window_decoding_results,
        args.bootstraps,
        condition_comparisons,
        rois,
        percentile=args.percentile,
        cluster_percentile=args.cluster_percentile,
        n_cluster_perms=args.n_cluster_perms,
        random_state=args.random_state,
        unit_of_analysis=args.unit_of_analysis
    )
    
    sub_str = str(len(args.subjects))
    
    analysis_params_str = (
            f"{sub_str}_subs_{elec_string_to_add_to_filename}_{args.clf_model_str}_" 
            f"{args.bootstraps}boots_{args.n_splits}splits_{args.n_repeats}reps_"
            f"{args.unit_of_analysis}_unit_ev_{args.explained_variance}"
        )   
                
    master_results = {
        'stats': all_bootstrap_stats,
        'metadata': {
            'args': vars(args), # Save all arguments from the run
            'analysis_params_str': analysis_params_str
        },
        'comparison_clusters': {} # We will populate this in the loops below
    }
       
    # define color and linestyle for plotting true vs shuffle
    colors = {
        'true': '#0173B2',  # Blue
        'shuffle': '#949494'  # Gray
    }
    
    linestyles = {
        'true': '-',
        'shuffle': '--'
    }  
    
    # then plot using the pooled statistics
    for condition_comparison in condition_comparisons.keys():
        for roi in rois:
            if roi in all_bootstrap_stats[condition_comparison]:
                stats = all_bootstrap_stats[condition_comparison][roi] 
                time_window_centers = time_window_decoding_results[0][condition_comparison][roi]['time_window_centers']
                
                # extract the correct keys based on unit_of_analysis
                unit = stats['unit_of_analysis']
                
                plot_accuracies_nature_style(
                    time_points=time_window_centers,
                    accuracies_dict={
                        'true': stats[f'{unit}_true_accs'], # use the full distribution
                        'shuffle': stats[f'{unit}_shuffle_accs']
                    },
                    significant_clusters=stats['significant_clusters'],
                    window_size=args.window_size,
                    step_size=args.step_size,
                    sampling_rate=args.sampling_rate,
                    comparison_name=f'bootstrap_true_vs_shuffle_{condition_comparison}',
                    roi=roi,
                    save_dir=os.path.join(save_dir, f"{condition_comparison}", f"{roi}"),
                    timestamp=args.timestamp,
                    p_thresh=args.percentile,
                    colors=colors,
                    linestyles=linestyles,
                    single_column=args.single_column,
                    show_legend=args.show_legend,
                    ylim=(0.3, 1.0),
                    show_chance_level=False, # The pooled shuffle line is the new chance level 
                    filename_suffix=analysis_params_str  
                )    
                
    # pooled cm traces for debugging        
    run_debug_cm_traces(
        time_window_decoding_results=time_window_decoding_results,
        condition_comparisons=condition_comparisons,
        rois=rois,
        cats_by_roi=cats_by_roi,
        args=args,
        save_dir=save_dir,
        analysis_params_str=analysis_params_str,
    )
    
    # run context comparisons comparing true vs. true decoding and also true vs. shuffle decoding for lwpc, lwps, etc.         
    run_all_context_comparisons(
        args=args,
        time_window_decoding_results=time_window_decoding_results,
        all_bootstrap_stats=all_bootstrap_stats,
        master_results=master_results,
        rois=rois,
        save_dir=save_dir,
        analysis_params_str=analysis_params_str,
    )
                
    # --- Save all results to a single file ---
    results_filename = f"{args.timestamp}_MASTER_RESULTS_{analysis_params_str}_{args.condition_label}.pkl"
    
    results_save_path = os.path.join(save_dir, results_filename)
    
    # Try to grab time_window_centers and add to metadata
    try:
        first_comp = list(time_window_decoding_results[0].keys())[0]
        first_roi = list(time_window_decoding_results[0][first_comp].keys())[0]
        twc = time_window_decoding_results[0][first_comp][first_roi]['time_window_centers']
        master_results['metadata']['time_window_centers'] = twc
    except Exception as e:
        print(f"Warning: Could not save time_window_centers to metadata. {e}")

    print(f"\n💾 Saving all statistical results to: {results_save_path}")
    with open(results_save_path, 'wb') as f:
        pickle.dump(master_results, f)

    print("\n✅ Analysis and saving complete.")
                  
if __name__ == "__main__":
    # This block is only executed when someone runs this script directly
    # Since your run script calls main() directly, this block won't be executed
    # But we'll keep it minimal for compatibility
    
    # Check if being called with SimpleNamespace (from run script)
    import sys
    if len(sys.argv) == 1:
        # No command line arguments, must be imported and called from run script
        pass
    else:
        # Someone is trying to run this directly with command line args
        print("This script should be called via run_decoding.py")
        print("Direct command-line execution is not supported with complex parameters.")
        sys.exit(1)
